In [1]:
!python -m pip install --upgrade pip --default-timeout=300

In [2]:
!python -m pip install mlflow --timeout 1200 --retries 10 --no-cache-dir

In [3]:
!pip install awscli

In [4]:
!pip install boto3

In [5]:
!pip install python-dotenv

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# !aws configure set aws_access_key_id os.getenv("aws_access_key_id")
# !aws configure set aws_secret_access_key os.getenv("aws_secret_access_key")
# !aws configure set region "ap-south-1"

# uncomment above before running

In [32]:
import mlflow
# uncmment this before run
#mlflow.set_tracking_uri("http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/")

In [34]:
#uncomment this before run
#mlflow.set_experiment("Exp 3 TFIDF Trigram - Max features")

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow.sklearn

In [11]:
df = pd.read_csv("cleaned_df.csv")

In [12]:
df.columns

Index(['Unnamed: 0', 'clean_comment', 'category'], dtype='object')

In [13]:
df.drop(columns=['Unnamed: 0'], inplace=True)

In [15]:
df

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1
...,...,...
36788,jesus,0
36789,kya bhai pure saal chutiya banaya modi aur jab...,1
36790,downvote karna tha par upvote hogaya,0
36791,haha nice,1


In [19]:
df.isna().sum()

clean_comment    131
category           0
dtype: int64

In [20]:
df.dropna(inplace=True)

In [21]:
df.isnull().sum()

clean_comment    0
category         0
dtype: int64

In [28]:
def run_exp_tfidf_max_features(max_features):
    ngram_range = (1,3)

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

    X = df['clean_comment']
    Y = df['category']

    X = vectorizer.fit_transform(X)

    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=.2, random_state=42, stratify=Y)

    with mlflow.start_run() as run:
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("max_features", max_features)

        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)
        model.fit(x_train, y_train)

        y_pred = model.predict(x_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy",accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(cm, annot=True, cmap="Blues", fmt="d")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"confusion matrix: TF-IDF Trigram - max features= {max_features}")
        plt.savefig(f"confusion_matrix_tfidf(1,3)_{max_features}.png")
        mlflow.log_artifact(f"confusion_matrix_tfidf(1,3)_{max_features}.png")
        plt.close()

        mlflow.sklearn.log_model(
            model,
            name=f"RF_model_tfidf_trigram_{max_features}",
            skops_trusted_types=["sklearn.tree._tree.Tree"]
        )

    print(f"Accuracy_{max_features}: {accuracy}")


In [30]:
max_feature_values = [500, 700, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

for max_features in max_feature_values:
    run_exp_tfidf_max_features(max_features)

🏃 View run zealous-robin-717 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/3/runs/5ce26b8fe08440fdbf9ef84ce1d871fe
🧪 View experiment at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/3
Accuracy_500: 0.6660302741033683
🏃 View run lyrical-fowl-889 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/3/runs/5a23ba2f002d4f80a090fdc6f1b7587e
🧪 View experiment at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/3
Accuracy_700: 0.6608482203736533
🏃 View run shivering-eel-626 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/3/runs/8f44a53f5d254d82835ba391fb21da3e
🧪 View experiment at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/3
Accuracy_1000: 0.6646665757534433
🏃 View run serious-shark-637 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/3/runs/0cea17e3fd824fb0ac2a1571e5428f6